In [8]:
import requests
import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import io

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    average_precision_score
)
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [27]:
url = 'https://drive.google.com/uc?id=1r7avcqz1wm7_2NYb_gCraCiPYWZC1AxK'
df = pd.read_csv(url)
df.index = df.index + 1
df = df.drop(columns=['Id'])
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
1,6,148,72,35,0,33.6,0.627,50,1
2,1,85,66,29,0,26.6,0.351,31,0
3,8,183,64,0,0,23.3,0.672,32,1
4,1,89,66,23,94,28.1,0.167,21,0
5,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count}zeros ({zero_count/len(df)*100:.1f}%)")

  Glucose: 18 zeros (0.7%)
  BloodPressure: 125 zeros (4.5%)
  SkinThickness: 800 zeros (28.9%)
  Insulin: 1330 zeros (48.0%)
  BMI: 39 zeros (1.4%)


In [ ]:
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)
    
for col in zero_cols:
    median_non_diabetic = df_clean[df_clean["Outcome"] == 0][col].median()
    median_diabetic = df_clean[df_clean["Outcome"] == 1][col].median()
    df_clean.loc[(df_clean["Outcome"] == 0) & (df_clean[col].isna()), col] = median_non_diabetic
    df_clean.loc[(df_clean["Outcome"] == 1) & (df_clean[col].isna()), col] = median_diabetic

for col in zero_cols:
    zero_count = (df_clean[col] == 0).sum()
    print(f"  {col}: {zero_count} zeros")

  Glucose: 0 zeros
  BloodPressure: 0 zeros
  SkinThickness: 0 zeros
  Insulin: 0 zeros
  BMI: 0 zeros


In [31]:
# Feature Engineering
df_clean["Glucose_BMI_Interaction"] = df_clean["Glucose"] * df_clean["BMI"]
df_clean["Age_BMI_Interaction"] = df_clean["Age"] * df_clean["BMI"]
df_clean["Insulin_Glucose_Ratio"] = df_clean["Insulin"] / (df_clean["Glucose"] + 1)
df_clean["Metabolic_Score"] = (df_clean["Glucose"] / df_clean["Glucose"].max()) + \
                               (df_clean["BMI"] / df_clean["BMI"].max()) + \
                               (df_clean["Age"] / df_clean["Age"].max())
